# EmPath v2 — GNN Landmark Experiment
**Graph Attention Network on 468 MediaPipe FaceMesh landmarks**

Before running: Runtime → Change runtime type → T4 GPU

### Files needed in Google Drive (`MyDrive/EmPath_v2/`):
```
EmPath_v2/
├── Results/biosignals_hrv/all_67_hrv.csv       ← biosignal features
├── Results/landmarks_gnn/raw_coords.npz        ← generated by Step 1 below
└── SRC/preprocessing/evaluate_gnn_landmarks_loso.py
```

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND"}')
assert torch.cuda.is_available(), 'No GPU! Runtime → Change runtime type → T4 GPU'

In [ ]:
# ── Cell 2: Install torch-geometric ───────────────────────────────────────
# Pure-Python wheels — no CUDA version matching needed for our graph size (468 nodes)
!pip install torch-geometric -q
import torch_geometric
print(f'torch-geometric: {torch_geometric.__version__}')

In [ ]:
# ── Cell 3: Mount Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/EmPath_v2'
assert os.path.isdir(BASE_DIR), f'Folder not found: {BASE_DIR}'
print(f'Drive mounted. BASE_DIR = {BASE_DIR}')

In [ ]:
# ── Cell 4: (SKIP if raw_coords.npz already uploaded) ─────────────────────
# Run extract_landmarks_raw_coords.py locally first, then upload raw_coords.npz
# to Drive at: EmPath_v2/Results/landmarks_gnn/raw_coords.npz
#
# Check it exists:
npz_path = os.path.join(BASE_DIR, 'Results', 'landmarks_gnn', 'raw_coords.npz')
if os.path.exists(npz_path):
    import numpy as np
    d = np.load(npz_path, allow_pickle=True)
    print(f'raw_coords.npz found')
    print(f'  coords shape : {d["coords"].shape}')   # expect (N, 24, 468, 2)
    print(f'  subjects     : {len(set(d["subject_ids"]))}')
    print(f'  PA2 / PA3    : {(d["labels"]==0).sum()} / {(d["labels"]==1).sum()}')
else:
    print('raw_coords.npz NOT found.')
    print('Upload it to Drive first: EmPath_v2/Results/landmarks_gnn/raw_coords.npz')

In [ ]:
# ── Cell 5: Run GNN LOSO ───────────────────────────────────────────────────
# Patches BASE_DIR into the script at runtime — no edits to the .py file needed
import importlib, sys

# Add SRC/preprocessing to path so the script can be imported as a module
script_dir = os.path.join(BASE_DIR, 'SRC', 'preprocessing')
sys.path.insert(0, script_dir)
sys.path.insert(0, BASE_DIR)

# Override BASE_DIR before the script reads it
os.environ['EMPATH_BASE_DIR'] = BASE_DIR

# Run via exec so BASE_DIR override takes effect
script_path = os.path.join(script_dir, 'evaluate_gnn_landmarks_loso.py')
with open(script_path) as f:
    src = f.read()

# Patch the BASE_DIR line to use Drive path
src = src.replace(
    "BASE_DIR    = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))",
    f"BASE_DIR    = '{BASE_DIR}'"
)

exec(src, {'__name__': '__main__'})

In [ ]:
# ── Cell 6: Download results ───────────────────────────────────────────────
# Results auto-saved to Drive at EmPath_v2/Results/gnn_loso/
# Also download locally as backup:
from google.colab import files
results_dir = os.path.join(BASE_DIR, 'Results', 'gnn_loso')
for f in os.listdir(results_dir):
    fpath = os.path.join(results_dir, f)
    print(f'Downloading {f}...')
    files.download(fpath)